In [46]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"구매 전환율(양성 비율): {y.mean():.4f}")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

쇼핑 세션 데이터: 12,330행 × 18열
구매 전환율(양성 비율): 0.1547

→ 준비 완료. 이제 여러분 차례입니다.


문제 1. 혼동행렬로 0.89를 쪼갠다
정확도 0.8973은 "2,466건 중 약 2,213건을 맞혔다"는 뜻입니다. 그런데 맞힌 것의 대부분이 '안 산다' 였다면 이 정확도만으로 마케팅팀의 구매 세션 포착 목적을 충족했는지 판단할 수 없습니다. 네 칸으로 쪼개서 확인합니다.

[문제 1]
1) 지난 순서에서 채택한 모델을 학습합니다.
   RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
2) 테스트셋에 대한 혼동행렬을 구해 TN · FP · FN · TP를 각각 출력합니다.
3) 정확도 · 정밀도 · 재현율 · F1을 계산해 함께 출력합니다.
4) "실제 구매 세션 중 몇 건을 놓쳤는가"를 건수와 비율로 설명합니다.

In [47]:
# [C2] 문제 1. 혼동행렬로 0.89를 쪼갠다
# ⌨️ 문제 1 — 정확도 하나를 네 칸으로 쪼개기
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

# 여기에 코드를 작성하세요 (분리 → 학습 → 혼동행렬 → 지표 4종 → 놓친 사람 수)

# 분리

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 학습
model_90 = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42).fit(X_train, y_train)

pred_90 = model_90.predict(X_test)

# 혼동행렬
cm = confusion_matrix(y_test, pred_90)
# 지표 4종 
tn, fp, fn, tp = cm.ravel()      # 순서 주의: TN, FP, FN, TP

print(f"TN(실제 구매가 아니고 아니라고 예측) = {tn}명")
print(f"FP(실제 구매가 아닌데 맞다고 예측) = {fp}명")
print(f"FN(실제 구매인데 아니라고 예측) = {fn}명")
print(f"TP(실제 구매이고 맞다고 예측) = {tp}명")

print("혼동행렬 (행=실제, 열=예측)")
print(cm)
print(f"정확도 = (TP+TN)/(TP+TN+FP+FN) = ({tp}+{tn})/({tp}+{tn}+{fp}+{fn})")
precision_manual = tp / (tp + fp)
recall_manual = tp / (tp + fn)
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)
print(f"정밀도 = TP/(TP+FP) = {tp}/({tp}+{fp}) = {precision_manual:.4f}")
print(f"재현율 = TP/(TP+FN) = {tp}/({tp}+{fn}) = {recall_manual:.4f}")
print(f"F1     = 2PR/(P+R)                = {f1_manual:.4f}")
print()

print("실제 구매 세션 중 몇 건을 놓쳤는가?")
print(f"놓친 구매 세션 수 = {fn}명")
print(f"놓친 구매 세션 비율 = {fn/(fn+tp):.4f} (재현율의 보충)")

TN(실제 구매가 아니고 아니라고 예측) = 2002명
FP(실제 구매가 아닌데 맞다고 예측) = 82명
FN(실제 구매인데 아니라고 예측) = 186명
TP(실제 구매이고 맞다고 예측) = 196명
혼동행렬 (행=실제, 열=예측)
[[2002   82]
 [ 186  196]]
정확도 = (TP+TN)/(TP+TN+FP+FN) = (196+2002)/(196+2002+82+186)
정밀도 = TP/(TP+FP) = 196/(196+82) = 0.7050
재현율 = TP/(TP+FN) = 196/(196+186) = 0.5131
F1     = 2PR/(P+R)                = 0.5939

실제 구매 세션 중 몇 건을 놓쳤는가?
놓친 구매 세션 수 = 186명
놓친 구매 세션 비율 = 0.4869 (재현율의 보충)


[문제 2]
1) cross_validate로 다섯 지표를 한 번에 구합니다.
   accuracy · precision · recall · f1 · average_precision(AP)
2) 각각 평균 ± 표준편차로 출력합니다.
3) AP를 양성 비율과 비교하고, 참고로 ROC-AUC도 구해 함께 확인합니다.
4) 다섯 지표 중 표준편차가 가장 큰 것이 무엇인지, 왜 그런지 설명합니다.

In [48]:
# [C3] 문제 2. 지표 5종을 교차 검증으로 한 번에 잰다
# ⌨️ 문제 2 — cross_validate로 다섯 지표를 동시에
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score

# 여기에 코드를 작성하세요 (5겹 CV로 지표 5종 → 평균±표준편차 → 기준선 대비)

model_90 = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42).fit(X_train, y_train)

def cv_report(model, X, y):
    rows = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "average_precision"])
    cw_compare = pd.DataFrame(rows)
    print(cw_compare.round(4).to_string(index=False))
    print(f"정확도 평균 ± 표준편차 = {cw_compare['test_accuracy'].mean():.4f} ± {cw_compare['test_accuracy'].std():.4f}")
    print(f"정밀도 평균 ± 표준편차 = {cw_compare['test_precision'].mean():.4f} ± {cw_compare['test_precision'].std():.4f}")
    print(f"재현율 평균 ± 표준편차 = {cw_compare['test_recall'].mean():.4f} ± {cw_compare['test_recall'].std():.4f}")
    print(f"F1 평균 ± 표준편차 = {cw_compare['test_f1'].mean():.4f} ± {cw_compare['test_f1'].std():.4f}")
    print(f"평균 정밀도(AP) 평균 ± 표준편차 = {cw_compare['test_average_precision'].mean():.4f} ± {cw_compare['test_average_precision'].std():.4f}")
    return cw_compare

base_result = cv_report(model_90, X, y)

proba = model_90.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, proba)
print(f"ROC AUC = {roc_auc:.4f}")



 fit_time  score_time  test_accuracy  test_precision  test_recall  test_f1  test_average_precision
   2.1783      0.1718         0.8978          0.7295       0.5381   0.6193                  0.7400
   2.1837      0.1602         0.8970          0.7361       0.5197   0.6092                  0.6957
   2.2729      0.1630         0.8994          0.7264       0.5628   0.6342                  0.7355
   2.2776      0.2092         0.8998          0.7228       0.5733   0.6394                  0.7441
   2.9438      0.2592         0.8905          0.7014       0.5105   0.5909                  0.7086
정확도 평균 ± 표준편차 = 0.8969 ± 0.0038
정밀도 평균 ± 표준편차 = 0.7232 ± 0.0131
재현율 평균 ± 표준편차 = 0.5409 ± 0.0270
F1 평균 ± 표준편차 = 0.6186 ± 0.0196
평균 정밀도(AP) 평균 ± 표준편차 = 0.7248 ± 0.0214
ROC AUC = 0.9002


In [49]:
def cv_report(model, X, y):
    rows = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "average_precision"])
    cw_compare = pd.DataFrame(rows)
    print(cw_compare.round(4).to_string(index=False))
    print(f"정확도 평균 ± 표준편차 = {cw_compare['test_accuracy'].mean():.4f} ± {cw_compare['test_accuracy'].std():.4f}")
    print(f"정밀도 평균 ± 표준편차 = {cw_compare['test_precision'].mean():.4f} ± {cw_compare['test_precision'].std():.4f}")
    print(f"재현율 평균 ± 표준편차 = {cw_compare['test_recall'].mean():.4f} ± {cw_compare['test_recall'].std():.4f}")
    print(f"F1 평균 ± 표준편차 = {cw_compare['test_f1'].mean():.4f} ± {cw_compare['test_f1'].std():.4f}")
    print(f"평균 정밀도(AP) 평균 ± 표준편차 = {cw_compare['test_average_precision'].mean():.4f} ± {cw_compare['test_average_precision'].std():.4f}")
    return cw_compare

base_result = cv_report(model_90, X, y)
balanced_result = cv_report(cb_model_90, X, y)  

 fit_time  score_time  test_accuracy  test_precision  test_recall  test_f1  test_average_precision
   7.9650      0.5789         0.8978          0.7295       0.5381   0.6193                  0.7400
   7.8339      0.6021         0.8970          0.7361       0.5197   0.6092                  0.6957
   8.5005      0.5824         0.8994          0.7264       0.5628   0.6342                  0.7355
   7.5356      0.6180         0.8998          0.7228       0.5733   0.6394                  0.7441
   8.2802      0.5841         0.8905          0.7014       0.5105   0.5909                  0.7086
정확도 평균 ± 표준편차 = 0.8969 ± 0.0038
정밀도 평균 ± 표준편차 = 0.7232 ± 0.0131
재현율 평균 ± 표준편차 = 0.5409 ± 0.0270
F1 평균 ± 표준편차 = 0.6186 ± 0.0196
평균 정밀도(AP) 평균 ± 표준편차 = 0.7248 ± 0.0214
 fit_time  score_time  test_accuracy  test_precision  test_recall  test_f1  test_average_precision
   8.1139      0.6033         0.8739          0.5601       0.8556   0.6771                  0.7275
   8.1414      0.4786         0.8666      

재현율의 표준편차가 0.0270로 가장 큽니다 — 정확도(0.0038)의 7배가 넘습니다.

이유는 분모에 있습니다. 각 검증 겹에서 정확도는 약 2,466건 전체로 계산하지만, 재현율은 그중 약 382건인 양성 세션으로 계산합니다. 분모가 더 작으므로 겹별 구성 변화에 더 민감할 수 있습니다. 그래서 소수 클래스 지표를 보고할 때는 표준편차를 함께 적어야 합니다.

어느 지표로 보고할 것인가. 마케팅팀의 관심은 "실제 구매 세션을 얼마나 포착하는가"이므로 재현율과 AP가 주 지표이고, 쿠폰이 낭비되지 않는지를 보는 정밀도가 함께 갑니다. 정확도는 보조 지표로 제시하되 단독으로 결론을 내리지 않습니다.

[문제 3]
1) 임계값을 0.10부터 0.70까지 훑으며 양성 예측 건수 · 정밀도 · 재현율 · F1을 표로 만듭니다.
2) F1이 가장 높은 임계값을 찾습니다.
3) "쿠폰 500장" 제약을 만족하는 임계값을 역산합니다.
   (확률 상위 500건만 고르려면 임계값이 얼마여야 하는가)
4) 그 운영점에 실제 구매 세션이 몇 건 포함되는지 계산해 마케팅팀에 보고합니다.

what is "양성 예측 건수..?" 

In [50]:
# [C4] 문제 3. 임계값을 훑어 운영점을 정한다
# ⌨️ 문제 3 — 용량 제약(쿠폰 500장)을 임계값으로 번역하기

# 여기에 코드를 작성하세요 (임계값 표 → F1 최적 → 상위 500건 역산 → 성과 보고)

rows = []
for t in [0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:
    p_t = (proba >= t).astype(int) # <- 임계값 적용한 예측
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, p_t).ravel()
    rows.append({
        "임계값": t,
        "양성 예측 건수": p_t.sum(), # 이게 맞나
        "정확도": accuracy_score(y_test, p_t),
        "정밀도": precision_score(y_test, p_t, zero_division=0),
        "재현율": recall_score(y_test, p_t),
        "F1": f1_score(y_test, p_t, zero_division=0),
        "놓침(FN)": fn_t,
        "헛경보(FP)": fp_t,
        "실제 구매 세션 포함": tp_t
    })
sweep_t = pd.DataFrame(rows)
print(sweep_t.round(4).to_string(index=False))
print()
print(f"F1이 가장 높은 임계값 = {sweep_t.loc[sweep_t['F1'].idxmax(), '임계값']:.2f}")
print(f"상위 500건을 고르려면 임계값 = {np.sort(proba)[::-1][499]:.4f}")

 임계값  양성 예측 건수    정확도    정밀도    재현율     F1  놓침(FN)  헛경보(FP)  실제 구매 세션 포함
 0.7       112 0.8792 0.8750 0.2565 0.3968     284       14           98
 0.6       183 0.8933 0.8251 0.3953 0.5345     231       32          151
 0.5       278 0.8913 0.7050 0.5131 0.5939     186       82          196
 0.4       396 0.8881 0.6338 0.6571 0.6452     131      145          251
 0.3       482 0.8743 0.5747 0.7251 0.6412     105      205          277
 0.2       543 0.8674 0.5506 0.7827 0.6465      83      244          299
 0.1       729 0.8212 0.4595 0.8770 0.6031      47      394          335

F1이 가장 높은 임계값 = 0.20
상위 500건을 고르려면 임계값 = 0.2683


In [51]:
t = np.sort(proba)[::-1][499]
p_t = (proba >= t).astype(int) # <- 임계값 적용한 예측
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, p_t).ravel()
print(f"임계값 = {t:.4f}")
print(f"양성 예측 건수 = {p_t.sum()}"), # 이게 맞나
print(f"정확도 = {accuracy_score(y_test, p_t):.4f}"),
print(f"정밀도 = {precision_score(y_test, p_t, zero_division=0):.4f}"),
print(f"재현율 = {recall_score(y_test, p_t):.4f}"),
print(f"F1 = {f1_score(y_test, p_t, zero_division=0):.4f}"),
print(f"놓침(FN) = {fn_t}"),
print(f"헛경보(FP) = {fp_t}"),
print(f"실제 구매 세션 포함 = {tp_t}")



임계값 = 0.2683
양성 예측 건수 = 500
정확도 = 0.8727
정밀도 = 0.5680
재현율 = 0.7435
F1 = 0.6440
놓침(FN) = 98
헛경보(FP) = 216
실제 구매 세션 포함 = 284


[문제 4]
1) class_weight="balanced"를 준 같은 모델로 교차 검증 지표 5종을 다시 구합니다.
2) 문제 2의 기본 설정과 나란히 표로 비교합니다.
3) 홀드아웃 혼동행렬도 함께 구해 TP·FN이 어떻게 달라졌는지 확인합니다.
4) 채택할 것인지 결정하고, 그 이유를 적습니다.
   (임계값 조정으로도 같은 효과를 낼 수 있다는 점을 함께 고려합니다)

In [52]:
# [C5] 문제 4. `class_weight` 보정, 채택할 것인가
# ⌨️ 문제 4 — 보정 전후를 같은 방식으로 비교

# 여기에 코드를 작성하세요 (balanced CV 지표 → 기본과 비교표 → 혼동행렬 대조)

model_90 = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42).fit(X_train, y_train)
cb_model_90 = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, class_weight="balanced").fit(X_train, y_train)

print("기본 모델")
base_result = cv_report(model_90, X, y)
print("class_weight-balanced 모델")
balanced_result = cv_report(cb_model_90, X, y)

cb_pred_90 = cb_model_90.predict(X_test)

# 혼동행렬
cb_cm_90 = confusion_matrix(y_test, cb_pred_90)
# 지표 4종 
cb_tn, cb_fp, cb_fn, cb_tp = cb_cm_90.ravel()      # 순서 주의: TN, FP, FN, TP

print("혼동행렬 (행=실제, 열=예측)")
print(cb_cm_90)
print()
print(f"  TP(이탈을 제대로 찾음) = {cb_tp}")
print(f"  FP(헛경보)             = {cb_fp}")
print(f"  FN(놓침)               = {cb_fn}")
print(f"  TN(유지를 제대로 넘김) = {cb_tn}")
print()
print(f"→ 정확도 = (TP+TN)/전체 = ({cb_tp}+{cb_tn})/{len(y_test)} = {accuracy_score(y_test, cb_pred_90):.3f}")
print(f"  틀린 것은 {cb_fp + cb_fn}개이고, 헛경보 {cb_fp}개와 놓침 {cb_fn}개로 나뉩니다.")

기본 모델
 fit_time  score_time  test_accuracy  test_precision  test_recall  test_f1  test_average_precision
   4.3741      0.1593         0.8978          0.7295       0.5381   0.6193                  0.7400
   2.2337      0.1666         0.8970          0.7361       0.5197   0.6092                  0.6957
   2.2262      0.1740         0.8994          0.7264       0.5628   0.6342                  0.7355
   2.2003      0.1672         0.8998          0.7228       0.5733   0.6394                  0.7441
   2.1824      0.1707         0.8905          0.7014       0.5105   0.5909                  0.7086
정확도 평균 ± 표준편차 = 0.8969 ± 0.0038
정밀도 평균 ± 표준편차 = 0.7232 ± 0.0131
재현율 평균 ± 표준편차 = 0.5409 ± 0.0270
F1 평균 ± 표준편차 = 0.6186 ± 0.0196
평균 정밀도(AP) 평균 ± 표준편차 = 0.7248 ± 0.0214
class_weight-balanced 모델
 fit_time  score_time  test_accuracy  test_precision  test_recall  test_f1  test_average_precision
   2.1697      0.1646         0.8739          0.5601       0.8556   0.6771                  0.7275
   2.2313  

In [53]:
cb_model_90 = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, class_weight="balanced").fit(X_train, y_train)
cb_pred_90 = cb_model_90.predict(X_test)


# 혼동행렬
cb_cm = confusion_matrix(y_test, cb_pred_90)

cb_tn, cb_fp, cb_fn, cb_tp = cb_cm.ravel()      # 순서 주의: TN, FP, FN, TP

print("원래 혼동행렬 (행=실제, 열=예측)")
print(cm)
print(f"TN(실제 구매가 아니고 아니라고 예측) = {tn}명")
print(f"FP(실제 구매가 아닌데 맞다고 예측) = {fp}명")
print(f"FN(실제 구매인데 아니라고 예측) = {fn}명")
print(f"TP(실제 구매이고 맞다고 예측) = {tp}명")

print("class_weight-balanced 혼동행렬 (행=실제, 열=예측)")
print(cb_cm)
print(f"TN(실제 구매가 아니고 아니라고 예측) = {cb_tn}명")
print(f"FP(실제 구매가 아닌데 맞다고 예측) = {cb_fp}명")
print(f"FN(실제 구매인데 아니라고 예측) = {cb_fn}명")
print(f"TP(실제 구매이고 맞다고 예측) = {cb_tp}명")



원래 혼동행렬 (행=실제, 열=예측)
[[2002   82]
 [ 186  196]]
TN(실제 구매가 아니고 아니라고 예측) = 2002명
FP(실제 구매가 아닌데 맞다고 예측) = 82명
FN(실제 구매인데 아니라고 예측) = 186명
TP(실제 구매이고 맞다고 예측) = 196명
class_weight-balanced 혼동행렬 (행=실제, 열=예측)
[[1835  249]
 [  78  304]]
TN(실제 구매가 아니고 아니라고 예측) = 1835명
FP(실제 구매가 아닌데 맞다고 예측) = 249명
FN(실제 구매인데 아니라고 예측) = 78명
TP(실제 구매이고 맞다고 예측) = 304명


tp 196 -> 304로 예측 성공이 늘어남
fn 186 -> 78명으로 예측 실패가 줄어들었음

양성 예측 553명

**실제 구매를 잘 예측하게 됨.**
그러나 실제 구매가 아닌데 맞다고 예측하는 게 82->249로 헛경보가 늘었음.
전에는 놓침이 많았는데, 양성을 더 적극적으로 예측하게 만드는 보정을 하니 당연.
 


In [57]:
# [C6] 문제 5. 모델 카드 v4 완성 — 오늘의 제출물
# ⌨️ 문제 5 — 임계값 표를 파일로 남기기

# 여기에 코드를 작성하세요 (thr_tbl을 CSV로 저장하고 다시 읽어 확인)
df_row = pd.concat([base_result, balanced_result], ignore_index=True)
df_row.to_csv("c:\\Users\\tjwls\\OneDrive\\문서\\ai-data-bootcamp\\D021\\threshold_table_v4.csv")